In [4]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Wpisz dokładne nazwy swoich plików z pasmem 6 i 5
plik_swir = 'LC_clipped_band6.tif' 
plik_nir = 'LC_clipped_band5.tif'

with rasterio.open(plik_swir) as src_swir:
    swir = src_swir.read(1).astype('float32')
    transform = src_swir.transform
    crs = src_swir.crs
    width = src_swir.width
    height = src_swir.height

with rasterio.open(plik_nir) as src_nir:
    nir = src_nir.read(1).astype('float32')

np.seterr(divide='ignore', invalid='ignore')

# 1. AGRESYWNA MASKA TŁA (wyłapuje zera oraz wartości NaN)
maska_tla = (swir == 0) | np.isnan(swir) | (nir == 0) | np.isnan(nir)

# Tworzymy pustą macierz NDBI i obliczamy wartości tylko tam, gdzie są poprawne dane (Kraków)
ndbi = np.zeros_like(swir)
piksele_danych = ~maska_tla
ndbi[piksele_danych] = (swir[piksele_danych] - nir[piksele_danych]) / (swir[piksele_danych] + nir[piksele_danych])

# 2. SZTYWNE RAMY KONTRASTU (naprawa kolorów)
# Wartości NDBI zazwyczaj wahają się od -0.2 (woda/lasy) do ok. 0.4 (gęsty beton). 
# Wymuszamy te granice, żeby paleta idealnie ułożyła się na zabudowie.
vmin = -0.2
vmax = 0.4

# Normalizacja do przedziału 0-1
ndbi_norm = (ndbi - vmin) / (vmax - vmin)
ndbi_norm = np.clip(ndbi_norm, 0, 1)

# Definicja palety kolorów
kolory_ndbi = ['#e0e0e0', '#ffaa00', '#e60000', '#660000']
cmap = LinearSegmentedColormap.from_list('ndbi_palette', kolory_ndbi)

colored_image = cmap(ndbi_norm) # Wynik (H, W, 4)

# 3. ODCINANIE TŁA (idealne granice zamiast kwadratu)
# Ustawiamy kanał przezroczystości (Alpha = 0) wszędzie tam, gdzie jest tło
colored_image[maska_tla, 3] = 0.0 

ndbi_rgba = (colored_image * 255).astype('uint8')
ndbi_rgba_chw = np.transpose(ndbi_rgba, (2, 0, 1)) 

profil = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 4, # 4 kanały = obsługa przezroczystości
    'dtype': 'uint8',
    'crs': crs,
    'transform': transform,
    'compress': 'lzw'
}

with rasterio.open('Krakow_NDBI_RGB.tif', 'w', **profil) as dst:
    dst.write(ndbi_rgba_chw)

print("Sukces! Wygenerowano poprawiony, przezroczysty plik z wyraźnymi kolorami betonu.")

Sukces! Wygenerowano poprawiony, przezroczysty plik z wyraźnymi kolorami betonu.
